In [ ]:
# ==============================================================================
# Block 1: CNN Encoder + Autoregressive Transformer Decoder
# ==============================================================================

import torch
import torch.nn as nn


class AutoregressiveTokenDecoder(nn.Module):
    """
    用於固定 10 層薄膜反向設計的 Autoregressive Transformer。

    Input
    -----
    spectrum:
        Transmittance Spectrum
        shape = (batch_size, 601)

    tgt_tokens:
        已知的厚度 Token 序列
        shape = (batch_size, sequence_length)

    Output
    ------
    logits:
        每個位置對 292 個 Token 的預測結果
        shape = (batch_size, sequence_length, 292)

    Token 定義
    ----------
    Token 0:
        <BOS>

    Token 1~291:
        對應厚度 10~300 nm
    """

    def __init__(
        self,
        input_dim=601,
        num_layers=10,
        vocab_size=292,
        d_model=256,
        nhead=8,
        num_layers_dec=6,
        dim_feedforward=1024,
        dropout=0.05,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.num_layers = num_layers
        self.d_model = d_model

        # 1D-CNN Transmittance Spectrum 特徵抽取器
        # Input:  (B, 1, 601)
        # Output: (B, 256, 601)
        self.spec_conv = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Conv1d(128, d_model, kernel_size=3, padding=1),
        )

        # Transmittance Spectrum 的 wavelength positional embedding
        self.spec_pos_emb = nn.Parameter(torch.randn(1, input_dim, d_model) * 0.02)

        # Thickness Token Embedding
        self.token_emb = nn.Embedding(vocab_size, d_model)

        # Thickness sequence 的位置資訊，包含 <BOS>
        self.pos_emb = nn.Parameter(torch.randn(1, num_layers + 1, d_model) * 0.02)

        # Transformer Decoder
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
        )

        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers_dec)

        # 將 Decoder 輸出轉成 292 個 Token logits
        self.head = nn.Linear(d_model, vocab_size)


    def forward(self, spectrum, tgt_tokens):

        batch_size, seq_len = tgt_tokens.size()

        # Spectrum:
        # (B, 601) -> (B, 1, 601) -> (B, 256, 601) -> (B, 601, 256)
        spec_feats = self.spec_conv(spectrum.unsqueeze(1)).transpose(1, 2)
        spec_feats = spec_feats + self.spec_pos_emb

        # Thickness Token:
        # (B, seq_len) -> (B, seq_len, 256)
        tgt_emb = self.token_emb(tgt_tokens) + self.pos_emb[:, :seq_len, :]

        # 防止 Decoder 在預測目前位置時看到後面的正確答案
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(spectrum.device)

        # tgt: Thickness Token Sequence
        # memory: CNN 提取的 Transmittance Spectrum Features
        out = self.transformer_decoder(tgt=tgt_emb, memory=spec_feats, tgt_mask=causal_mask)

        # (B, seq_len, 256) -> (B, seq_len, 292)
        logits = self.head(out)

        return logits

In [ ]:
# ==============================================================================
# Block 2: Dataset Loading + Training
# ==============================================================================

import os
import glob
import sys
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm


"""
訓練參數設定。

DATA_DIR        : Dataset 資料夾位置
BATCH_SIZE      : 每個 Batch 的樣本數
EPOCHS          : 完整 Dataset 重複訓練次數
LEARNING_RATE   : AdamW 初始 Learning Rate
WEIGHT_DECAY    : AdamW Weight Decay
LABEL_SMOOTHING : Cross-Entropy Label Smoothing
MODEL_SAVE_PATH : 最佳模型權重儲存位置
"""

DATA_DIR = r"C:\Users\chuyu\OneDrive\Coding\Python\Project\Progress\Dataset\IntegerData2M"

BATCH_SIZE = 256
EPOCHS = 35
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05

MODEL_SAVE_PATH = "best_ar_token_decoder.pth"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 0 if os.name == "nt" else 4

print(f"Device: {DEVICE}")


class ThinFilmTokenDataset(Dataset):
    """
    讀取固定 10 層薄膜 Dataset。

    Dataset 中：
        thickness : shape = (N, 10)
        T         : shape = (N, 601)

    Thickness Token：
        Token 0     = <BOS>
        Token 1     = 10 nm
        Token 2     = 11 nm
        ...
        Token 291   = 300 nm

    輸出每筆資料：
        tokens      : shape = (10,)
        spectrum    : shape = (601,)
    """

    def __init__(self, data_dir, wl_min=200.0, wl_max=800.0):

        npz_files = sorted(glob.glob(os.path.join(data_dir, "**", "Data_*.npz"), recursive=True))

        if len(npz_files) == 0:
            raise FileNotFoundError(f"在 {data_dir} 找不到 Data_*.npz")

        all_thickness = []
        all_transmittance = []
        self.wl_nm = None

        for file_path in npz_files:
            npz_data = np.load(file_path)

            all_thickness.append(npz_data["thickness"])
            all_transmittance.append(npz_data["T"])

            if self.wl_nm is None:
                self.wl_nm = npz_data["wl_nm"]

        self.raw_thickness = np.concatenate(all_thickness, axis=0).astype(np.float32)
        self.full_transmittance = np.concatenate(all_transmittance, axis=0).astype(np.float32)

        # 10~300 nm → Token 1~291，Token 0 保留給 <BOS>
        clamped_thickness = np.clip(self.raw_thickness, 10.0, 300.0)
        self.token_labels = (np.round(clamped_thickness - 10.0) + 1).astype(np.int64)

        # 選擇使用的波長範圍
        crop_mask = (self.wl_nm >= wl_min) & (self.wl_nm <= wl_max)
        self.transmittance = self.full_transmittance[:, crop_mask]

        print(f"Dataset samples: {len(self.raw_thickness):,}")
        print(f"Spectrum dimension: {self.transmittance.shape[1]}")

    def __len__(self):
        return len(self.raw_thickness)

    def __getitem__(self, idx):
        tokens = torch.tensor(self.token_labels[idx], dtype=torch.long)
        spectrum = torch.tensor(self.transmittance[idx], dtype=torch.float32)

        return tokens, spectrum


"""
載入 Dataset 並建立 DataLoader。

DataLoader 每次輸出：
    batch_tokens   : shape = (BATCH_SIZE, 10)
    batch_spectrum : shape = (BATCH_SIZE, 601)
"""

dataset = ThinFilmTokenDataset(data_dir=DATA_DIR, wl_min=200.0, wl_max=800.0)

train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=(DEVICE.type == "cuda"),
    num_workers=NUM_WORKERS,
)


"""
建立 Block 1 定義的模型。

模型輸入：
    Transmittance Spectrum : (B, 601)

模型輸出：
    logits : (B, 10, 292)
"""

model = AutoregressiveTokenDecoder(input_dim=dataset.transmittance.shape[1]).to(DEVICE)


"""
訓練設定。

Loss:
    Cross-Entropy Loss

Optimizer:
    AdamW

Learning Rate Scheduler:
    Cosine Annealing

AMP:
    CUDA 環境下使用混合精度訓練
"""

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

total_steps = EPOCHS * len(train_loader)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=total_steps,
    eta_min=1e-6,
)

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))


"""
Training。

Teacher Forcing：

Target：
    [d1, d2, d3, ..., d10]

Decoder Input：
    [BOS, d1, d2, ..., d9]

模型依序學習：
    BOS              → d1
    BOS, d1          → d2
    BOS, d1, d2      → d3
    ...
    BOS, d1, ..., d9 → d10
"""

loss_history = []
accuracy_history = []
best_loss = float("inf")

print("\n" + "=" * 60)
print(f"Start Training | Epochs: {EPOCHS} | Samples: {len(dataset):,}")
print("=" * 60)


for epoch in range(1, EPOCHS + 1):

    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc=f"Epoch [{epoch:02d}/{EPOCHS:02d}]", file=sys.stdout)

    for batch_tokens, batch_spectrum in pbar:

        batch_tokens = batch_tokens.to(DEVICE, non_blocking=True)
        batch_spectrum = batch_spectrum.to(DEVICE, non_blocking=True)

        bos = torch.zeros((batch_tokens.size(0), 1), dtype=torch.long, device=DEVICE)

        tgt_input = torch.cat([bos, batch_tokens[:, :-1]], dim=1)
        tgt_target = batch_tokens

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            logits = model(batch_spectrum, tgt_input)
            loss = criterion(logits.reshape(-1, model.vocab_size), tgt_target.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        predictions = torch.argmax(logits, dim=-1)

        correct += (predictions == tgt_target).sum().item()
        total += tgt_target.numel()
        total_loss += loss.item() * batch_spectrum.size(0)

        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            accuracy=f"{correct / total:.3f}",
        )

    avg_loss = total_loss / len(dataset)
    accuracy = correct / total

    loss_history.append(avg_loss)
    accuracy_history.append(accuracy)

    current_lr = scheduler.get_last_lr()[0]

    print(
        f"Epoch [{epoch:02d}/{EPOCHS:02d}] | "
        f"Loss: {avg_loss:.5f} | "
        f"Token Accuracy: {accuracy * 100:.2f}% | "
        f"LR: {current_lr:.2e}"
    )

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)

        print(f"Saved best model | Loss: {best_loss:.5f}")


print("\nTraining Finished")


"""
Training Loss Curve。
"""

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(loss_history) + 1), loss_history, marker="o")

plt.title("Training Cross-Entropy Loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# Block 3: Export Trained Model
# ==============================================================================

import torch

# 讀取 Block 2 儲存的最佳模型權重
best_state_dict = torch.load(
    MODEL_SAVE_PATH,
    map_location="cpu"
)

# 將模型架構設定與訓練完成的權重一起儲存
export_payload = {
    "model_config": {
        "input_dim": 601,
        "num_layers": 10,
        "vocab_size": 292,
        "d_model": 256,
        "nhead": 8,
        "num_layers_dec": 6,
        "dim_feedforward": 1024,
        "dropout": 0.05,
    },

    "state_dict": best_state_dict,
}

save_path = "thin_film_transformer_final.pth"

torch.save(export_payload, save_path)

print(f"Model exported to: {save_path}")